# 農地問題エリア検出システム

**画像ソース**: Google Maps Static API（Google Earthと同等の高解像度衛星画像）

**ワークフロー**
1. GeoJSON × Google Maps Static API → 高解像度JPEG保存
2. 手動ラベリング（farmland / problem に仕分け）
3. CNN学習（EfficientNet-B0）
4. 未ラベル画像を推論・自動仕分け
5. 手動仕分け → 継続学習（Active Learning）

---
## 0. セットアップ

### Google Maps Static API キーの取得手順
1. [Google Cloud Console](https://console.cloud.google.com/) を開く
2. プロジェクト作成 → 「APIとサービス」→「ライブラリ」
3. 「Maps Static API」を検索して有効化
4. 「APIとサービス」→「認証情報」→「APIキーを作成」
5. 下の `GOOGLE_MAPS_API_KEY` にコピー

**料金**: 月1000リクエストまで無料。以降 $2/1000リクエスト（1万枚で約$18）

In [ ]:
# ライブラリのインストール（初回のみ）
!pip install geopandas shapely Pillow torch torchvision tqdm matplotlib requests scikit-learn pandas pyproj -q

In [ ]:
import hashlib
import io
import math
import os
import shutil
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {DEVICE}')

In [ ]:
# ディレクトリ作成
for d in ['data/unlabeled', 'data/farmland', 'data/problem', 'data/review', 'models', 'logs']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('ディレクトリ作成完了')

---
## 1. 衛星画像の取得（Google Maps Static API）

各農地ポリゴンの中心座標 + 適切なズームレベルで高解像度衛星画像を取得し、  
ポリゴン境界（緑線）を重ねてJPEGとして保存する。

In [ ]:
# ===== 設定 =====
GOOGLE_MAPS_API_KEY = 'YOUR_API_KEY_HERE'   # ← APIキーをここに貼る

GEOJSON_PATH  = 'data/farmland.geojson'     # ← 農地GeoJSONのパス
OUTPUT_DIR    = 'data/unlabeled'
MAX_POLYGONS  = 200     # None で全件。まず小さい数でテスト
IMG_SIZE      = 640     # 640が無料上限（Static API の最大サイズ）
ZOOM          = 18      # 18=約0.6m/pixel, 19=約0.3m/pixel, 20=約0.15m/pixel
                        # ポリゴンが大きければ自動で下げる（AUTO_ZOOM=True時）
AUTO_ZOOM     = True    # ポリゴンサイズに応じてズームを自動調整

In [ ]:
# ----- Google Maps Static API ヘルパー関数 -----

def meters_per_pixel(zoom, lat):
    """指定ズームレベルでの1ピクセルあたりのメートル数。"""
    return 156543.03392 * math.cos(math.radians(lat)) / (2 ** zoom)


def calc_zoom(bounds, img_px=640, padding=1.3):
    """
    バウンディングボックスが img_px に収まる最大ズームレベルを返す。
    padding: ポリゴンが画像端に近すぎないよう余白係数（>1で少し引き）
    """
    minx, miny, maxx, maxy = bounds
    lat_c = (miny + maxy) / 2
    # 経度方向のメートル幅
    deg_per_m_lon = 1.0 / (111320 * math.cos(math.radians(lat_c)))
    deg_per_m_lat = 1.0 / 110574
    span_m = max(
        (maxx - minx) / deg_per_m_lon,
        (maxy - miny) / deg_per_m_lat,
    ) * padding
    mpp_needed = span_m / img_px
    zoom = int(math.log2(156543.03392 * math.cos(math.radians(lat_c)) / mpp_needed))
    return max(1, min(zoom, 20))


def fetch_google_maps_image(lat, lon, zoom, size, api_key):
    """
    Google Maps Static API から衛星画像を取得して PIL Image を返す。
    """
    url = (
        'https://maps.googleapis.com/maps/api/staticmap'
        f'?center={lat},{lon}'
        f'&zoom={zoom}'
        f'&size={size}x{size}'
        f'&maptype=satellite'
        f'&key={api_key}'
    )
    for attempt in range(4):
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()
            img = Image.open(io.BytesIO(resp.content)).convert('RGB')
            return img
        except Exception as e:
            if attempt == 3:
                raise RuntimeError(f'画像取得失敗: {e}')
            time.sleep(2 ** attempt)


def latlon_to_pixel_google(lat, lon, center_lat, center_lon, zoom, img_px):
    """
    Google Maps の Mercator 投影に基づいて (lat, lon) を画像ピクセル座標に変換。
    中心座標 (center_lat, center_lon) を img_px//2 とする。
    """
    scale = 2 ** zoom

    def to_world(la, lo):
        x = (lo + 180) / 360 * 256 * scale
        sin_lat = math.sin(math.radians(la))
        y = (0.5 - math.log((1 + sin_lat) / (1 - sin_lat)) / (4 * math.pi)) * 256 * scale
        return x, y

    cx, cy = to_world(center_lat, center_lon)
    px, py = to_world(lat, lon)
    half = img_px / 2
    return int(half + (px - cx)), int(half + (py - cy))


def draw_polygon_overlay(img, geom, center_lat, center_lon, zoom):
    """Mercator 投影で正確にポリゴン境界を重ねる。"""
    overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    img_px = img.size[0]

    def ring_to_pixels(ring):
        return [
            latlon_to_pixel_google(lat, lon, center_lat, center_lon, zoom, img_px)
            for lon, lat in ring.coords
        ]

    rings = []
    if geom.geom_type == 'Polygon':
        rings = [geom.exterior] + list(geom.interiors)
    elif geom.geom_type == 'MultiPolygon':
        for poly in geom.geoms:
            rings.append(poly.exterior)
            rings.extend(poly.interiors)

    for ring in rings:
        pixels = ring_to_pixels(ring)
        if len(pixels) >= 3:
            draw.polygon(pixels, fill=(0, 200, 0, 55))
            draw.line(pixels + [pixels[0]], fill=(255, 255, 0, 230), width=2)

    return Image.alpha_composite(img.convert('RGBA'), overlay).convert('RGB')


def polygon_uid(geom, idx):
    return hashlib.md5(f'{idx}_{geom.wkt[:200]}'.encode()).hexdigest()[:12]


print('関数定義完了')

In [ ]:
# APIキーの確認（1枚だけ取得してテスト）
assert GOOGLE_MAPS_API_KEY != 'YOUR_API_KEY_HERE', 'APIキーを設定してください'

test_img = fetch_google_maps_image(35.6895, 139.6917, zoom=18, size=640, api_key=GOOGLE_MAPS_API_KEY)
plt.figure(figsize=(5, 5))
plt.imshow(test_img)
plt.title(f'テスト画像（東京 zoom=18, {test_img.size[0]}x{test_img.size[1]}px）')
plt.axis('off')
plt.show()
print('APIキー正常動作')

In [ ]:
# GeoJSON 読み込み
gdf = gpd.read_file(GEOJSON_PATH)
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)

if MAX_POLYGONS:
    gdf = gdf.head(MAX_POLYGONS)

print(f'処理対象ポリゴン数: {len(gdf)}')
print(f'推定リクエスト数: {len(gdf)} 件 / 料金目安: 無料枠1000件超過後 ${len(gdf)/1000*2:.2f}')
gdf.plot(figsize=(10, 8), color='green', alpha=0.3, edgecolor='black', linewidth=0.3)
plt.title('農地ポリゴン分布')
plt.show()

In [ ]:
# 画像取得ループ
output_dir = Path(OUTPUT_DIR)
success, skip, error = 0, 0, 0
meta_records = []

for idx, row in tqdm(gdf.iterrows(), total=len(gdf), desc='画像取得'):
    geom = row.geometry
    uid = polygon_uid(geom, idx)
    out_path = output_dir / f'{uid}.jpg'

    if out_path.exists():
        skip += 1
        continue

    bounds = geom.bounds  # (minx, miny, maxx, maxy)
    span = max(bounds[2] - bounds[0], bounds[3] - bounds[1])

    # 極端なサイズはスキップ
    if span < 0.00005 or span > 0.5:
        skip += 1
        continue

    # 中心座標
    centroid = geom.centroid
    center_lat, center_lon = centroid.y, centroid.x

    # ズームレベルの決定
    zoom = calc_zoom(bounds, img_px=IMG_SIZE) if AUTO_ZOOM else ZOOM

    try:
        img = fetch_google_maps_image(center_lat, center_lon, zoom, IMG_SIZE, GOOGLE_MAPS_API_KEY)
        img = draw_polygon_overlay(img, geom, center_lat, center_lon, zoom)
        img.save(out_path, 'JPEG', quality=92)
        success += 1
        meta_records.append({
            'uid': uid, 'path': str(out_path),
            'lat': center_lat, 'lon': center_lon, 'zoom': zoom,
        })
        # API レート制限対策（無料枠: 毎秒50リクエストまで）
        time.sleep(0.05)
    except Exception as e:
        tqdm.write(f'  [skip] idx={idx}: {e}')
        error += 1

pd.DataFrame(meta_records).to_csv(output_dir / 'metadata.csv', index=False)
print(f'\n完了: 成功={success}, スキップ={skip}, エラー={error}')

In [ ]:
# 取得した画像をサムネイル表示（先頭12枚）
images = sorted(Path(OUTPUT_DIR).glob('*.jpg'))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, p in zip(axes.flat, images):
    ax.imshow(Image.open(p))
    meta = pd.read_csv(Path(OUTPUT_DIR) / 'metadata.csv')
    row = meta[meta.uid == p.stem]
    zoom_val = int(row.zoom.values[0]) if len(row) else '?'
    ax.set_title(f'{p.stem[:8]}\nzoom={zoom_val}', fontsize=7)
    ax.axis('off')
for ax in axes.flat[len(images):]:
    ax.axis('off')
plt.suptitle('取得した農地画像（先頭12枚） - Google Maps 衛星画像')
plt.tight_layout()
plt.show()

---
## 2. 手動ラベリング（このセルを実行する前に手動作業）

`data/unlabeled/` の画像をファインダー/エクスプローラーで開き、  
目視で以下に振り分けてください（各クラス **50〜100枚以上** 推奨）:

- `data/farmland/` ← 農地が正常に写っている画像
- `data/problem/`  ← 建物・道路が1/3以上を占める画像

振り分け後、下のセルでデータ数を確認してから学習に進んでください。

In [ ]:
# ラベリング状況の確認
for cls in ['farmland', 'problem', 'unlabeled', 'review']:
    n = len(list(Path(f'data/{cls}').glob('*.jpg'))) if Path(f'data/{cls}').exists() else 0
    print(f'  {cls:12s}: {n} 枚')

---
## 3. CNN 学習

In [ ]:
# ===== 学習設定 =====
DATA_DIR    = 'data'
MODEL_OUT   = 'models/model_v1.pth'
EPOCHS      = 20
BATCH_SIZE  = 16   # GPU なし / メモリ少なければ 8 に下げる
LR          = 1e-4
VAL_RATIO   = 0.15

In [ ]:
def build_transforms(train=True):
    if train:
        return T.Compose([
            T.Resize((224, 224)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.RandomRotation(15),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def build_model(num_classes=2):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total


print('モデル関数定義完了')

In [ ]:
# データセット準備
full_ds = datasets.ImageFolder(DATA_DIR, transform=build_transforms(train=True))
print(f'クラス: {full_ds.class_to_idx}')
print(f'総サンプル数: {len(full_ds)}')

n_val = max(1, int(len(full_ds) * VAL_RATIO))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))

val_ds_clean = datasets.ImageFolder(DATA_DIR, transform=build_transforms(train=False))
val_ds.dataset = val_ds_clean

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'train={n_train}枚, val={n_val}枚')

In [ ]:
# 学習実行
model = build_model().to(DEVICE)

counts = [0] * 2
for _, label in full_ds.samples:
    counts[label] += 1
class_weights = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
best_val_acc = 0.0
Path(MODEL_OUT).parent.mkdir(parents=True, exist_ok=True)

for epoch in tqdm(range(1, EPOCHS + 1), desc='学習'):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion)
    scheduler.step()
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    tqdm.write(f'Epoch {epoch:03d} | train acc={train_acc:.3f} loss={train_loss:.4f} | val acc={val_acc:.3f} loss={val_loss:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'class_to_idx': full_ds.class_to_idx, 'val_acc': val_acc}, MODEL_OUT)
        tqdm.write(f'  → モデル保存 (val_acc={val_acc:.4f})')

print(f'\n学習完了。最良 val_acc={best_val_acc:.4f}')

In [ ]:
# 学習曲線
df_hist = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(df_hist.epoch, df_hist.train_loss, label='train')
ax1.plot(df_hist.epoch, df_hist.val_loss,   label='val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(df_hist.epoch, df_hist.train_acc, label='train')
ax2.plot(df_hist.epoch, df_hist.val_acc,   label='val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')
ax2.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('logs/training_curve.png', dpi=150)
plt.show()

---
## 4. 推論・自動仕分け

学習済みモデルで `data/unlabeled/` を推論。  
- 確信度 ≥ `THRESHOLD` → 自動で farmland / problem に振り分け  
- 確信度 < `THRESHOLD` → `data/review/` へ（手動確認用）

In [ ]:
# ===== 推論設定 =====
PREDICT_MODEL = 'models/model_v1.pth'
PREDICT_INPUT = 'data/unlabeled'
THRESHOLD     = 0.85
DRY_RUN       = True   # True=移動しない（確認用）。実行時は False に

In [ ]:
# モデル読み込み
state = torch.load(PREDICT_MODEL, map_location=DEVICE)
pred_model = build_model().to(DEVICE)
pred_model.load_state_dict(state['model'])
pred_model.eval()
idx_to_class = {v: k for k, v in state['class_to_idx'].items()}
print(f'モデル読み込み完了 (val_acc={state["val_acc"]:.4f})')
print(f'クラスマップ: {idx_to_class}')

In [ ]:
# 推論実行
infer_transform = build_transforms(train=False)
input_dir = Path(PREDICT_INPUT)
out_dirs = {
    'farmland': input_dir.parent / 'farmland',
    'problem':  input_dir.parent / 'problem',
    'review':   input_dir.parent / 'review',
}
if not DRY_RUN:
    for d in out_dirs.values():
        d.mkdir(parents=True, exist_ok=True)

images = sorted(input_dir.glob('*.jpg'))
records = []

with torch.no_grad():
    for img_path in tqdm(images, desc='推論'):
        try:
            img = Image.open(img_path).convert('RGB')
            tensor = infer_transform(img).unsqueeze(0).to(DEVICE)
            probs = F.softmax(pred_model(tensor), dim=1)[0]
            pred_idx = probs.argmax().item()
            confidence = probs[pred_idx].item()
            pred_class = idx_to_class[pred_idx]
            dest = pred_class if confidence >= THRESHOLD else 'review'
            records.append({'file': img_path.name, 'pred': pred_class,
                            'confidence': round(confidence, 4), 'dest': dest})
            if not DRY_RUN:
                shutil.move(str(img_path), out_dirs[dest] / img_path.name)
        except Exception as e:
            tqdm.write(f'  [skip] {img_path.name}: {e}')

df_pred = pd.DataFrame(records)
df_pred.to_csv('data/predict_report.csv', index=False)
print('\n--- 推論結果 ---')
print(df_pred.dest.value_counts().to_string())
print(f'\n平均確信度: {df_pred.confidence.mean():.4f}')
if DRY_RUN:
    print('\n※ DRY_RUN=True のため実際には移動していません。')

In [ ]:
# 確信度分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_pred.confidence, bins=30, edgecolor='black')
axes[0].axvline(THRESHOLD, color='red', linestyle='--', label=f'threshold={THRESHOLD}')
axes[0].set_xlabel('確信度'); axes[0].set_ylabel('件数')
axes[0].set_title('確信度分布'); axes[0].legend()
df_pred.dest.value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('振り分け結果'); axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# review/ の画像をサムネイル表示（手動仕分けの参考に）
review_images = sorted(Path('data/review').glob('*.jpg'))[:16]
if review_images:
    cols = 4
    rows = (len(review_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    for ax, p in zip(axes.flat, review_images):
        row = df_pred[df_pred.file == p.name]
        conf = row.confidence.values[0] if len(row) else 0
        pred = row.pred.values[0] if len(row) else '?'
        ax.imshow(Image.open(p))
        ax.set_title(f'pred={pred}\nconf={conf:.2f}', fontsize=8)
        ax.axis('off')
    for ax in axes.flat[len(review_images):]:
        ax.axis('off')
    plt.suptitle('要確認画像（review/）→ farmland/ or problem/ に移動してください')
    plt.tight_layout()
    plt.show()
else:
    print('review/ に画像がありません')

---
## 5. Active Learning（継続学習）

**実行前に**: `data/review/` の画像を `data/farmland/` または `data/problem/` に手動で移動してください。

In [ ]:
# ===== 継続学習設定 =====
BASE_MODEL    = 'models/model_v1.pth'
NEW_MODEL_OUT = 'models/model_v2.pth'
RETRAIN_EPOCHS     = 10
RETRAIN_BATCH_SIZE = 16
RETRAIN_LR         = 5e-5

In [ ]:
# データ状況チェック
review_n   = len(list(Path('data/review').glob('*.jpg')))
farmland_n = len(list(Path('data/farmland').glob('*.jpg')))
problem_n  = len(list(Path('data/problem').glob('*.jpg')))
print(f'farmland: {farmland_n} 枚')
print(f'problem:  {problem_n} 枚')
print(f'review:   {review_n} 枚', '← まだ仕分けが残っています！' if review_n > 0 else '(仕分け済み)')
if review_n > 0:
    print('\n⚠️  review/ に画像が残っています。移動してから再実行してください。')
elif farmland_n < 10 or problem_n < 10:
    print('\n⚠️  学習データが少なすぎます（各クラス10枚以上必要）')
else:
    print('\n✓ 継続学習を実行できます。次のセルを実行してください。')

In [ ]:
# 継続学習
retrain_ds = datasets.ImageFolder(DATA_DIR, transform=build_transforms(train=True))
n_val_r = max(1, int(len(retrain_ds) * VAL_RATIO))
n_train_r = len(retrain_ds) - n_val_r
train_ds_r, val_ds_r = random_split(retrain_ds, [n_train_r, n_val_r], generator=torch.Generator().manual_seed(42))
val_ds_r.dataset = datasets.ImageFolder(DATA_DIR, transform=build_transforms(train=False))

train_loader_r = DataLoader(train_ds_r, batch_size=RETRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
val_loader_r   = DataLoader(val_ds_r,   batch_size=RETRAIN_BATCH_SIZE, shuffle=False, num_workers=0)

retrain_model = build_model().to(DEVICE)
retrain_state = torch.load(BASE_MODEL, map_location=DEVICE)
retrain_model.load_state_dict(retrain_state['model'])
print(f'起点モデル val_acc={retrain_state["val_acc"]:.4f}')

counts_r = [0] * 2
for _, label in retrain_ds.samples:
    counts_r[label] += 1
weights_r = torch.tensor([1.0 / c for c in counts_r], dtype=torch.float).to(DEVICE)
criterion_r = nn.CrossEntropyLoss(weight=weights_r)
optimizer_r = torch.optim.AdamW(retrain_model.parameters(), lr=RETRAIN_LR, weight_decay=1e-4)
scheduler_r = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_r, T_max=RETRAIN_EPOCHS)

best_val_acc_r = 0.0
Path(NEW_MODEL_OUT).parent.mkdir(parents=True, exist_ok=True)

for epoch in tqdm(range(1, RETRAIN_EPOCHS + 1), desc='継続学習'):
    train_loss_r, train_acc_r = run_epoch(retrain_model, train_loader_r, criterion_r, optimizer_r)
    val_loss_r,   val_acc_r   = run_epoch(retrain_model, val_loader_r,   criterion_r)
    scheduler_r.step()
    tqdm.write(f'Epoch {epoch:03d} | train acc={train_acc_r:.3f} | val acc={val_acc_r:.3f}')
    if val_acc_r > best_val_acc_r:
        best_val_acc_r = val_acc_r
        torch.save({'epoch': epoch, 'model': retrain_model.state_dict(),
                    'class_to_idx': retrain_ds.class_to_idx, 'val_acc': val_acc_r}, NEW_MODEL_OUT)
        tqdm.write(f'  → モデル保存: {NEW_MODEL_OUT}')

print(f'\n継続学習完了。{retrain_state["val_acc"]:.4f} → {best_val_acc_r:.4f}')

---
## 次のサイクル

精度が上がるまで繰り返す:
```
セクション4（推論）→ 手動仕分け → セクション5（継続学習）
```
`PREDICT_MODEL` と `BASE_MODEL` を最新バージョン（例: `model_v2.pth`）に更新して実行してください。